# Student Pass/Fail Prediction

This is a **classification** project. We have data about students and we want to build a model that predicts whether a student **passes** (1) or **fails** (0).

Goal: learn the standard workflow that most machine learning projects follow:

1. Load the data
2. Explore the data
3. Separate features (X) and target (y)
4. Split into train/test
5. Train a model
6. Evaluate the model
7. Make predictions

## Step 1: Load the data

First we read the CSV file into a `DataFrame` using **pandas** and take a quick look at it.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv("../dataset/Pass-Fail Data.csv")
df.head()

,student_id,attendance_pct,homework_pct,midterm_score,study_hours_per_week,pass
0,1,95,92,88,12,1
1,2,88,85,79,10,1
2,3,60,55,58,4,0
3,4,72,70,65,6,1
4,5,40,45,50,3,0


## Step 2: Explore the data

Before training a model we need to understand the data:

- How big is it?
- What types of columns do we have?
- Is there any missing data?
- How balanced are the classes (pass vs fail)?

In [2]:
print("Shape (rows, columns):", df.shape)
print()
print("Column data types:")
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())

Shape (rows, columns): (100, 6)

Column data types:
student_id              int64
attendance_pct          int64
homework_pct            int64
midterm_score           int64
study_hours_per_week    int64
pass                    int64
dtype: object

Missing values per column:
student_id              0
attendance_pct          0
homework_pct            0
midterm_score           0
study_hours_per_week    0
pass                    0
dtype: int64


In [3]:
print("Statistical summary of numeric columns:")
df.describe(include="all")

Statistical summary of numeric columns:


,student_id,attendance_pct,homework_pct,midterm_score,study_hours_per_week,pass
count,100.000000,100.000000,100.00000,100.000000,100.00000,100.000000
mean,50.500000,69.520000,69.03000,68.780000,7.28000,0.600000
std,29.011492,17.651783,17.01304,14.717254,3.62115,0.492366
min,1.000000,30.000000,35.00000,42.000000,2.00000,0.000000
25%,25.750000,55.000000,55.00000,56.000000,4.00000,0.000000
50%,50.500000,72.500000,70.00000,68.000000,7.00000,1.000000
75%,75.250000,85.000000,85.00000,82.000000,10.00000,1.000000
max,100.000000,95.000000,96.00000,97.000000,15.00000,1.000000


## Step 3: Separate features (X) and target (y)

The model needs to learn the relationship between the **inputs** and the **answer**:

- `X` = what we know before the result → `attendance_pct`, `homework_pct`, `midterm_score`, `study_hours_per_week`
- `y` = what we want to predict → `pass`

Why we drop columns:

- `pass` is the **answer** — if the model can see it, it will just memorize it (data leakage) and never learn a real rule.
- `student_id` is just a row number — it carries no meaning and teaches the model nothing.

In [4]:
# Features (inputs)
X = df.drop(columns=["pass", "student_id"])

# Target (answer)
y = df["pass"]

print("X shape (rows, features):", X.shape)
print("y shape (rows,):", y.shape)
print()
print("Feature columns:")
print(X.columns.tolist())
print()
print("Target value counts:")
print(y.value_counts())

X shape (rows, features): (100, 4)
y shape (rows,): (100,)

Feature columns:
['attendance_pct', 'homework_pct', 'midterm_score', 'study_hours_per_week']

Target value counts:
pass
1    60
0    40
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(80, 4)
(80,)
(20, 4)
(20,)


## Step 4: Split data into train and test

A model must be evaluated on data it has **never seen** during training — otherwise it just scores high from memorization (overfitting).

So we shuffle all 100 rows and split them:

- **80 rows -> train** → the model learns from these
- **20 rows -> test** → held back, we pretend we've never seen them, and measure how well it learned

- `test_size=0.2` → 20% of the data is held back for testing
- `random_state=42` → fixes the shuffle so the split is the same every run (reproducible)

## Step 5: Train a model (Logistic Regression)

Now the actual **training** — we give the model the training data and it learns the rule.

```python
model.fit(X_train, y_train)
```

- `LogisticRegression()` → creates an empty (untrained) classifier
- `.fit(X_train, y_train)` → the learning step: it studies the patterns in the training rows and their answers

After `fit`, the model holds the learned rules inside itself and is ready to make predictions on new data.

In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Training complete.")
print("Learned coefficients:", model.coef_)

Training complete.
Learned coefficients: [[0.59167298 0.73545576 0.28241582 0.31162228]]


In [7]:
y_pred = model.predict(X_test)
print(y_pred,"This is new one")
print(y_test)

[1 1 0 0 1 1 0 0 0 1 1 1 0 1 1 0 1 0 1 1] This is new one
83    1
53    1
70    0
45    0
44    1
39    1
22    0
80    0
10    0
0     1
18    1
30    1
73    0
33    1
90    1
4     0
76    1
77    0
12    1
31    1
Name: pass, dtype: int64


In [8]:
accuray = accuracy_score(y_test,y_pred)
print("Accuracy:", accuray)


Accuracy: 1.0


## Step 6: Save the trained model

Right now the model only exists in the notebook's memory. If we close it or restart the kernel, it disappears and we'd have to retrain.

In real projects we save the model to a file using **joblib**, so we can reload it later and use it to make predictions **without retraining**.

In [9]:
import joblib

model_filename = "../models/student_pass_fail_model.pkl"
joblib.dump(model, model_filename)

print("Model saved to:", model_filename)
print("File size:", round(len(open(model_filename, "rb").read()) / 1024, 2), "KB")

Model saved to: ../models/student_pass_fail_model.pkl
File size: 1.19 KB


### How to reload the model later (in a new session)

```python
import joblib
model = joblib.load("../models/student_pass_fail_model.pkl")
model.predict(X_test)   # immediate predictions, no training needed
```